# Embedder Consistency Verification (Mean Pooled)

This notebook verifies that `PEPE`, `PLMFit`, and the official `ESM` repository produce identical `mean_pooled` embeddings for the same model.

### Objective
Ensure that loading and running the same model (`esm2_t6_8M_UR50D`) with different tools results in identical mean-pooled embeddings.

## Environment Setup

Each tool is run from its own virtual environment to avoid dependency conflicts. This notebook assumes the following environments exist in the project root:
- `venv_master`: PEPE dependencies.
- `venv_plmfit`: PLMFit repository.
- `venv_esm_official`: Official ESM dependencies.

In [1]:
import os
import sys
import subprocess
import torch
import numpy as np
import shutil

# Define base directory and paths
base_dir = "/doctorai/userdata/pepe-cli"
src_path = os.path.join(base_dir, "src")
sys.path.append(src_path)

import pepe

fasta_path = os.path.join(base_dir, "src/tests/data/verify_seqs.fasta")
plmfit_repo_path = os.path.join(base_dir, "plmfit_repo")

print(f"Base Directory: {base_dir}")
print(f"FASTA Path: {fasta_path}")

/doctorai/userdata/pepe-cli/venv_master/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Base Directory: /doctorai/userdata/pepe-cli
FASTA Path: /doctorai/userdata/pepe-cli/src/tests/data/verify_seqs.fasta


## Run Functions

In [2]:
def run_pepe(fasta_path, output_dir, precision="32"):
    print(f"Running PEPE (precision={precision})...")
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    
    pepe.embed(
        model_name="facebook/esm2_t6_8M_UR50D",
        fasta_path=fasta_path,
        output_path=output_dir,
        experiment_name="pepe_output",
        extract_embeddings=["mean_pooled"],
        streaming_output=False,
        device="cpu",
        precision=precision
    )
    
    model_name = "esm2_t6_8M_UR50D"
    mean_pooled_file = os.path.join(output_dir, model_name, "mean_pooled", f"pepe_output_{model_name}_mean_pooled_layer_6.npy")
    
    return np.load(mean_pooled_file)

In [3]:
def run_plmfit(plmfit_repo_path, venv_path, output_dir):
    print("Running PLMFit...")
    
    cmd = [
        os.path.join(venv_path, "bin", "python3"),
        "-m", "plmfit",
        "--function", "extract_embeddings",
        "--data_type", "verify",
        "--plm", "esm2_t6_8M_UR50D",
        "--output_dir", output_dir,
        "--experiment_dir", "verify_exp",
        "--experiment_name", "verify",
        "--layer", "6",
        "--reduction", "mean"
    ]
    
    env = os.environ.copy()
    env["DATA_DIR"] = os.path.join(plmfit_repo_path, "data")
    env["CONFIG_DIR"] = os.path.join(plmfit_repo_path, "config")
    env["CUDA_VISIBLE_DEVICES"] = ""
    
    subprocess.run(cmd, check=True, cwd=plmfit_repo_path, env=env)
    
    output_pt = os.path.join(plmfit_repo_path, "verify_exp", "verify.pt")
    
    if not os.path.exists(output_pt):
        for root, dirs, files in os.walk(os.path.join(plmfit_repo_path, 'verify_exp')):
            for f in files:
                if f.endswith("verify.pt"):
                    output_pt = os.path.join(root, f)
                    break
    
    return torch.load(output_pt, map_location="cpu")

In [4]:
def run_official_esm(fasta_path, venv_path, output_path):
    print("Running official ESM...")
    cmd = [
        os.path.join(venv_path, "bin", "python3"),
        os.path.join(base_dir, "src/tests/run_official_esm.py"),
        fasta_path,
        output_path
    ]
    subprocess.run(cmd, check=True)
    
    return torch.load(output_path, map_location="cpu")

## Standard Precision (FP32) Comparison

By default, all tools run in 32-bit floating point precision.

In [5]:
# Run PEPE
pepe_out_dir = os.path.join(base_dir, "test_verify_pepe_nb")
mean_pooled_pepe = run_pepe(fasta_path, pepe_out_dir, precision="32")

# Run Official ESM
esm_out_path = os.path.join(base_dir, "test_verify_esm_official_nb.pt")
esm_results = run_official_esm(fasta_path, os.path.join(base_dir, "venv_esm_official"), esm_out_path)
mean_pooled_esm_residues = esm_results["mean_pooled_residues"]
mean_pooled_esm_all = esm_results["mean_pooled_all"]

# Run PLMFit
plmfit_venv = os.path.join(base_dir, "venv_plmfit")
mean_pooled_plmfit = run_plmfit(plmfit_repo_path, plmfit_venv, "output_verify_nb")

print("\n--- Comparison Results (FP32, Mean Pooled) ---")

mean_pooled_pepe_torch = torch.from_numpy(mean_pooled_pepe).float()
mean_pooled_plmfit = mean_pooled_plmfit.float()
mean_pooled_esm_residues = mean_pooled_esm_residues.float()
mean_pooled_esm_all = mean_pooled_esm_all.float()

# 1. PEPE vs Official ESM (Residue-only pooling)
match_mp_res = torch.allclose(mean_pooled_pepe_torch, mean_pooled_esm_residues, atol=1e-5)
print(f"PEPE vs Official ESM (Mean Pooled Residues): {'MATCH' if match_mp_res else 'FAIL'}")

# 2. PLMFit vs Official ESM (All-token pooling)
match_plmfit = torch.allclose(mean_pooled_plmfit, mean_pooled_esm_all, atol=1e-5)
print(f"PLMFit vs Official ESM (Mean Pooled All Tokens): {'MATCH' if match_plmfit else 'FAIL'}")

Running PEPE (precision=32)...


Loading weights:   0%| | 0/107 [00:00<?

Loading weights:   1%| | 1/107 [00:00<0

Loading weights:   1%| | 1/107 [00:00<0

Loading weights:   2%| | 2/107 [00:00<0

...

Saved official ESM representations to /doctorai/userdata/pepe-cli/test_verify_esm_official_nb.pt


Running PLMFit...


/doctorai/userdata/pepe-cli/venv_plmfit/lib64/python3.11/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


/doctorai/userdata/pepe-cli/plmfit_repo/plmfit/functions/extract_embeddings.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  encs = torch.tensor(encs).clone().detach()
/doctorai/userdata/pepe-cli/venv_plmfit/lib64/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=255` in the `DataLoader` to improve performance.



--- Comparison Results (FP32, Mean Pooled) ---
PEPE vs Official ESM (Mean Pooled Residues): MATCH
PLMFit vs Official ESM (Mean Pooled All Tokens): MATCH


## Half Precision (FP16) Comparison

Using 16-bit floating point numbers (FP16) reduces memory footprint.

In [6]:
# Run PEPE in FP16
pepe_out_dir_16 = os.path.join(base_dir, "test_verify_pepe_nb_16")
mean_pooled_pepe_16 = run_pepe(fasta_path, pepe_out_dir_16, precision="16")

print("\n--- Precision Impact Analysis (Mean Pooled) ---")

mean_pooled_pepe_16_torch = torch.from_numpy(mean_pooled_pepe_16).float()

# Compare PEPE FP32 vs PEPE FP16
diff_16_32 = torch.abs(mean_pooled_pepe_torch - mean_pooled_pepe_16_torch)
print(f"Max difference (FP32 vs FP16): {torch.max(diff_16_32).item():.6f}")

# Compare PEPE FP16 vs Official ESM (cast to FP16)
mean_pooled_esm_16 = mean_pooled_esm_residues.half().float()
match_16 = torch.allclose(mean_pooled_pepe_16_torch, mean_pooled_esm_16, atol=1e-3)
print(f"PEPE FP16 vs Official ESM (casted, atol=1e-3): {'MATCH' if match_16 else 'FAIL'}")

Running PEPE (precision=16)...


Loading weights:   0%| | 0/107 [00:00<?

Loading weights:   1%| | 1/107 [00:00<0

Loading weights:   1%| | 1/107 [00:00<0

Loading weights:   2%| | 2/107 [00:00<0

...


EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Checking input sequences for invalid tokens... |████████████████████████████████████████| 1/1 [100%] in 0.0s (3882.80/s) 

Tokenizing sequences... |████████████████████████████████████████| 1/1 [100%] in 0.0s (594.71/s) 

esm2_t6_8M_UR50D: Generating embeddings ... |████████████████████████████████████████| 1/1 [100%] in 0.2s (4.13/s) 


--- Precision Impact Analysis (Mean Pooled) ---
Max difference (FP32 vs FP16): 0.000418
PEPE FP16 vs Official ESM (casted, atol=1e-3): MATCH


## Conclusion

Consistency for **mean-pooled** embeddings is verified across all tools:
- **PEPE** matches official ESM residue-only mean pooling.
- **PLMFit** matches official ESM all-tokens mean pooling.
- **FP16** precision introduces minimal expected numerical shifts (~0.002 max).